# Notebook 17 — Gradient Direction Hypothesis: Is the Damage Direction-Aligned?
**CS 590NN | Amogh | Apr 25 — final novel experiment**

## Hypothesis

NB11 said the optimizer step is the cause. NB15 said in-place modification is required. Neither tells us **what property of the update** destroys A.

**NB17 hypothesis:** The damage to A is proportional to the **cosine alignment** between B's weight-delta direction and the weight-delta direction that was applied during edit A.

In parameter space: if B pushes the weights along the same axis that A pushed them, A gets overwritten. If B's update is orthogonal to A's, A survives.

This is a deeper mechanistic claim than "in-place modification" — it specifies which in-place modifications are dangerous.

## Method

For each pair (A, B):
1. Snapshot original weights `W_orig`
2. Edit A → snapshot `W_postA`. Compute `delta_A = W_postA - W_orig` per MLP layer
3. Restore to `W_orig`. Edit B alone → snapshot `W_postB_solo`. Compute `delta_B_solo = W_postB_solo - W_orig`
4. **Compute cosine(delta_A, delta_B_solo) per MLP layer, average across shared layers**
5. Restore to `W_postA`. Edit B sequentially → measure `A_displaced`
6. Test correlation: does `cosine(delta_A, delta_B_solo)` predict `A_displaced`?

## Why this is novel

The editing literature doesn't measure cosine alignment between sequential edit weight deltas. Continual learning measures gradient cosine between *tasks*, but no analog for *edit weight deltas* in localized model editing.

## Predictions

| ρ value | Interpretation |
|---|---|
| ρ > +0.4, p < 0.05 | **Direction-aligned damage** — geometry of the update matters. Novel mechanism. |
| Weak/null (\|ρ\| < 0.2) | Update direction doesn't matter. Magnitude alone drives damage. |
| ρ < -0.4 | Counterintuitive: orthogonal updates damage more. Would need explanation. |

## Compute

~20 min A100. 10 pairs × 3 edit phases (A solo, B solo, A then B sequential).

### 17.0 Install (run once, restarts)

In [ ]:
import subprocess, sys, os
def pip(args): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + args)
pip(['numpy==1.26.4'])
pip(['transformer-lens', 'transformers', 'datasets', 'accelerate', 'einops', 'jaxtyping'])
pip(['matplotlib', 'scipy'])
print('Restarting...'); os.kill(os.getpid(), 9)

### 17.1 Imports + auto-fetch v3 pair indices

In [ ]:
import torch, json, requests, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from scipy import stats
from transformer_lens import HookedTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path('results_nb17'); RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)} ({free/1e9:.1f}/{total/1e9:.1f} GB free)')

REPO_RAW = 'https://raw.githubusercontent.com/rukmini-17/targeted-unlearning/main/circuit_pipeline/results'
V3_URL = f'{REPO_RAW}/sequential_editing_v3/sequential_edit_rows_v3_20260424_220951.json'
v3_data = requests.get(V3_URL, timeout=60).json()
v3_ok = [r for r in v3_data['rows'] if r.get('status') == 'ok']
seen = set()
PAIR_INDICES = []
for r in v3_ok:
    pid = r['pair_idx']
    if pid in seen: continue
    seen.add(pid)
    PAIR_INDICES.append((r['A_idx'], r['B_idx']))
    if len(PAIR_INDICES) >= 10: break
print(f'Selected {len(PAIR_INDICES)} pairs')

### 17.2 Load model + samples

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False, device=DEVICE,
)
model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token

@dataclass
class EditSample:
    idx: int; prompt: str; subject: str
    target_new: str; target_true: str

raw = requests.get('https://rome.baulab.info/data/dsets/counterfact.json', timeout=120).json()
def make_sample(idx):
    rr = raw[idx]['requested_rewrite']
    return EditSample(idx=idx, prompt=rr['prompt'].format(rr['subject']),
                      subject=rr['subject'],
                      target_new=' '+rr['target_new']['str'], target_true=' '+rr['target_true']['str'])

pairs = [{'A': make_sample(a), 'B': make_sample(b)} for a, b in PAIR_INDICES]
print(f'Loaded model + {len(pairs)} pairs')

### 17.3 ROME-trace + helpers (same as NB11/NB15)

In [ ]:
def rome_trace_top_layers(model, sample, k=5):
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0].item()
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0].item()
    tokens = model.to_tokens(sample.prompt)
    n_tok = tokens.shape[1]
    if n_tok <= 2: return list(range(min(k, model.cfg.n_layers)))
    subj_pos = list(range(1, n_tok-1))
    corrupt = tokens.clone()
    corrupt[0, subj_pos] = torch.randint(1000, model.cfg.d_vocab-1, (len(subj_pos),), device=tokens.device)
    with torch.no_grad():
        cl, cc = model.run_with_cache(tokens)
        kl, _ = model.run_with_cache(corrupt)
    clean_score = (cl[0,-1,true_id] - cl[0,-1,new_id]).item()
    corrupt_score = (kl[0,-1,true_id] - kl[0,-1,new_id]).item()
    eff = max(abs(clean_score - corrupt_score), 0.5)
    effects = []
    for L in range(model.cfg.n_layers):
        hn = f'blocks.{L}.hook_mlp_out'
        if hn not in cc: continue
        ca = cc[hn].clone()
        def hk(v, hook): return ca
        with torch.no_grad():
            patched = model.run_with_hooks(corrupt, fwd_hooks=[(hn, hk)])
        recovery = (patched[0,-1,true_id] - patched[0,-1,new_id]).item() - corrupt_score
        effects.append((L, abs(recovery)/eff))
    effects.sort(key=lambda x: -x[1])
    del cc; torch.cuda.empty_cache()
    return [L for L, _ in effects[:k]]

NEUTRAL = [
    'The sum of two and three is', 'Twelve divided by four equals',
    'The square root of nine is', 'Ten times ten equals',
    'The capital of Japan is', 'The largest ocean on Earth is the',
    'Mount Everest is located in the', 'The Amazon River flows through',
    'Water boils at one hundred degrees', 'The chemical symbol for gold is',
    'Plants produce oxygen through a process called', 'The Earth orbits the',
    'A week contains seven', 'The primary colors are red, blue, and',
    'Humans have two lungs and one', 'A triangle has three',
]

def cache_ref(model, prompts):
    cache = []
    model.eval()
    with torch.no_grad():
        for p in prompts:
            t = model.to_tokens(p)
            ref_lp = torch.log_softmax(model(t)[0,-1,:], dim=-1).detach().clone()
            cache.append((t, ref_lp))
    return cache

def kl_against(model, ref_cache):
    total = 0.0
    for t, ref_lp in ref_cache:
        edit_lp = torch.nn.functional.log_softmax(model(t)[0,-1,:], dim=-1)
        total = total + (ref_lp.exp() * (ref_lp - edit_lp)).sum()
    return total / len(ref_cache)

def edit_normal(model, sample, mlp_layers, n_steps, lr, beta_kl, ref_cache):
    params = []
    for L in mlp_layers:
        params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    if not params:
        for L in range(model.cfg.n_layers):
            params += [model.blocks[L].mlp.W_in, model.blocks[L].mlp.W_out]
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    opt = torch.optim.Adam(params, lr=lr)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0,0]
    tokens = model.to_tokens(sample.prompt)
    for _ in range(n_steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        logits = model(tokens)
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
        loss = -lp[new_id] + lp[true_id]
        if ref_cache and beta_kl > 0:
            loss = loss + beta_kl * kl_against(model, ref_cache)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        opt.step()
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()

def measure_p_new(model, sample):
    model.eval()
    with torch.no_grad():
        new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0]
        return float(torch.softmax(model(model.to_tokens(sample.prompt))[0,-1,:], dim=-1)[new_id])

def restore(model, state):
    with torch.no_grad():
        for n, p in model.named_parameters(): p.copy_(state[n])
    torch.cuda.empty_cache()

print('Helpers ready.')

### 17.4 Snapshot weight delta extraction

In [ ]:
def snapshot_mlp_weights(model, mlp_layers):
    """Capture current W_in, W_out for given layers as a dict."""
    snap = {}
    for L in mlp_layers:
        snap[L] = {
            'W_in':  model.blocks[L].mlp.W_in.detach().clone().cpu(),
            'W_out': model.blocks[L].mlp.W_out.detach().clone().cpu(),
        }
    return snap

def compute_delta(snap_after, snap_before):
    """Compute weight delta per layer."""
    delta = {}
    for L in snap_after:
        delta[L] = {
            'W_in':  snap_after[L]['W_in']  - snap_before[L]['W_in'],
            'W_out': snap_after[L]['W_out'] - snap_before[L]['W_out'],
        }
    return delta

def cosine_between_deltas(delta_A, delta_B):
    """
    Compute average cosine similarity between two weight delta dicts.
    Per-layer cosine across both W_in and W_out, then averaged across shared layers.
    """
    shared_layers = sorted(set(delta_A.keys()) & set(delta_B.keys()))
    if len(shared_layers) == 0:
        return float('nan'), 0
    
    cos_per_layer = []
    for L in shared_layers:
        # Concatenate W_in and W_out flat vectors per layer for full-MLP cosine
        a = torch.cat([delta_A[L]['W_in'].flatten(), delta_A[L]['W_out'].flatten()])
        b = torch.cat([delta_B[L]['W_in'].flatten(), delta_B[L]['W_out'].flatten()])
        a_norm = a.norm() + 1e-12
        b_norm = b.norm() + 1e-12
        cos = float((a @ b) / (a_norm * b_norm))
        cos_per_layer.append(cos)
    
    return float(np.mean(cos_per_layer)), len(shared_layers)

def delta_magnitude(delta):
    """L2 norm of the entire delta (across all layers, both W_in and W_out)."""
    total = 0.0
    for L in delta:
        total += float(delta[L]['W_in'].norm()**2 + delta[L]['W_out'].norm()**2)
    return float(np.sqrt(total))

print('Delta extraction utilities ready.')

### 17.5 Run the 3-phase protocol per pair

In [ ]:
N_STEPS_A = 5
N_STEPS_B = 3
BETA_KL = 0.1
LR = 5e-3

ref_cache = cache_ref(model, NEUTRAL)

all_samples = {s.idx: s for p in pairs for s in (p['A'], p['B'])}
print(f'ROME-trace for {len(all_samples)} samples...')
circuits = {idx: rome_trace_top_layers(model, s, k=5) for idx, s in all_samples.items()}
print('Done.')

print('\nSnapshotting original weights...')
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()

ROWS_FILE = RESULTS_DIR / f'cosine_geometry_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
rows = []
started = datetime.now()

for p_idx, p in enumerate(pairs):
    A, B = p['A'], p['B']
    layers_A = circuits[A.idx]
    layers_B = circuits[B.idx]
    union_layers = sorted(set(layers_A) | set(layers_B))
    
    try:
        # ===== Phase 1: snapshot original W on union layers =====
        snap_orig = snapshot_mlp_weights(model, union_layers)
        
        # ===== Phase 2: edit A solo, snapshot W_postA, compute delta_A =====
        edit_normal(model, A, layers_A, n_steps=N_STEPS_A, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        snap_postA = snapshot_mlp_weights(model, union_layers)
        delta_A = compute_delta(snap_postA, snap_orig)
        p_A_postA = measure_p_new(model, A)
        
        # ===== Phase 3: restore, edit B solo, snapshot W_postB_solo, compute delta_B_solo =====
        restore(model, original_state)
        edit_normal(model, B, layers_B, n_steps=N_STEPS_B, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        snap_postB_solo = snapshot_mlp_weights(model, union_layers)
        delta_B_solo = compute_delta(snap_postB_solo, snap_orig)
        p_B_solo = measure_p_new(model, B)
        
        # ===== Compute cosine and magnitudes =====
        cos_AB, n_shared_layers = cosine_between_deltas(delta_A, delta_B_solo)
        mag_A = delta_magnitude(delta_A)
        mag_B = delta_magnitude(delta_B_solo)
        
        # ===== Phase 4: restore to W_postA, run sequential edit B, measure A_displaced =====
        # First restore to original, then redo edit A so we're at W_postA
        restore(model, original_state)
        edit_normal(model, A, layers_A, n_steps=N_STEPS_A, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        # Now edit B sequentially
        edit_normal(model, B, layers_B, n_steps=N_STEPS_B, lr=LR, beta_kl=BETA_KL, ref_cache=ref_cache)
        p_A_postAB = measure_p_new(model, A)
        p_B_postAB = measure_p_new(model, B)
        A_displaced = max(0, p_A_postA - p_A_postAB)
        
        row = {
            'pair_idx': p_idx, 'A_idx': A.idx, 'B_idx': B.idx,
            'n_layers_A': len(layers_A), 'n_layers_B': len(layers_B),
            'n_shared_layers': n_shared_layers,
            'cosine_delta_A_delta_B_solo': cos_AB,
            'magnitude_delta_A': mag_A,
            'magnitude_delta_B_solo': mag_B,
            'p_A_postA': p_A_postA,
            'p_B_solo': p_B_solo,
            'p_A_postAB': p_A_postAB,
            'p_B_postAB': p_B_postAB,
            'A_displaced': A_displaced,
            'status': 'ok',
        }
        rows.append(row)
        print(f'[{p_idx+1}/{len(pairs)}] pair {p_idx}: cos(δA,δB)={cos_AB:+.3f}  '
              f'|δA|={mag_A:.2f}  |δB|={mag_B:.2f}  shared_layers={n_shared_layers}  '
              f'A_displaced={A_displaced:.3f}')
        with open(ROWS_FILE, 'w') as f: json.dump({'rows': rows}, f, indent=2)
    except Exception as e:
        print(f'[{p_idx+1}/{len(pairs)}] FAILED: {type(e).__name__}: {e}')
        import traceback; traceback.print_exc()
        rows.append({'pair_idx': p_idx, 'status': 'failed', 'error': str(e)[:200]})
        torch.cuda.empty_cache()

elapsed = (datetime.now() - started).total_seconds() / 60
print(f'\nTotal runtime: {elapsed:.1f} min')
restore(model, original_state)

### 17.6 Test the hypothesis

In [ ]:
df = pd.DataFrame([r for r in rows if r.get('status') == 'ok'])
print(f'OK rows: {len(df)}')
print('\nDescriptive stats:')
print(df[['cosine_delta_A_delta_B_solo', 'magnitude_delta_A', 'magnitude_delta_B_solo',
         'A_displaced', 'n_shared_layers']].describe().round(3))

print('\n=== HEADLINE TEST: cosine(δA, δB_solo) vs A_displaced ===')
clean = df[df['n_shared_layers'] >= 1].copy()
print(f'Clean rows (≥1 shared layer): {len(clean)}')

if len(clean) >= 5:
    rho, p = stats.spearmanr(clean['cosine_delta_A_delta_B_solo'], clean['A_displaced'])
    pearson_r, pearson_p = stats.pearsonr(clean['cosine_delta_A_delta_B_solo'], clean['A_displaced'])
    sig_s = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
    sig_p = '***' if pearson_p<0.001 else ('**' if pearson_p<0.01 else ('*' if pearson_p<0.05 else 'ns'))
    print(f'  Spearman rho = {rho:+.3f}  p = {p:.4f}  {sig_s}')
    print(f'  Pearson r    = {pearson_r:+.3f}  p = {pearson_p:.4f}  {sig_p}')
    
    print('\n=== Control: magnitude predictors ===')
    for col in ['magnitude_delta_A', 'magnitude_delta_B_solo']:
        rho2, p2 = stats.spearmanr(clean[col], clean['A_displaced'])
        sig2 = '***' if p2<0.001 else ('**' if p2<0.01 else ('*' if p2<0.05 else 'ns'))
        print(f'  {col:>26}: rho={rho2:+.3f}  p={p2:.4f}  {sig2}')
    
    # Multivariate: does cosine add over magnitudes?
    print('\n=== Multivariate OLS: A_displaced ~ cosine + |δA| + |δB| ===')
    from numpy.linalg import lstsq
    cols = ['cosine_delta_A_delta_B_solo', 'magnitude_delta_A', 'magnitude_delta_B_solo']
    sub = clean.dropna(subset=cols + ['A_displaced'])
    X = np.column_stack([np.ones(len(sub)), sub[cols].values])
    y = sub['A_displaced'].values
    beta, _, _, _ = lstsq(X, y, rcond=None)
    yp = X @ beta
    r2 = 1 - ((y-yp)**2).sum() / ((y-y.mean())**2).sum()
    coefs = ', '.join([f'{c.split("_")[0]}={beta[i+1]:+.4f}' for i, c in enumerate(cols)])
    print(f'  R²={r2:.3f}  intercept={beta[0]:+.3f}  {coefs}')
    
    # Verdict
    print('\n=== VERDICT ===')
    if rho > 0.4 and p < 0.05:
        print('  >>> HYPOTHESIS SUPPORTED: cosine(δA, δB) significantly predicts A_displaced.')
        print('  >>> Damage is direction-aligned in parameter space.')
        print('  >>> NOVEL MECHANISTIC CLAIM: which-direction-the-update-pushes matters.')
    elif rho < -0.4 and p < 0.05:
        print('  >>> HYPOTHESIS REVERSED: more aligned = less damage. Counterintuitive.')
        print('  >>> Possible explanation: aligned updates reinforce the same representation.')
    elif p < 0.05:
        print(f'  >>> Significant but weak (rho={rho:+.3f}). Direction matters slightly.')
    else:
        print('  >>> Hypothesis NOT supported. Cosine alignment does not predict damage.')
        print('  >>> Damage is direction-INDEPENDENT in parameter space.')
        print('  >>> Implies magnitude or other property dominates.')

### 17.7 Money plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(clean['cosine_delta_A_delta_B_solo'], clean['A_displaced'],
           s=80, alpha=0.7, color='#cc3333', edgecolors='black', lw=0.4)
for _, r in clean.iterrows():
    ax.annotate(f"{int(r['pair_idx'])}", (r['cosine_delta_A_delta_B_solo'], r['A_displaced']),
                fontsize=8, alpha=0.7, ha='center')
ax.set_xlabel('cosine(δA, δB_solo)  [parameter-space alignment]')
ax.set_ylabel('A_displaced')
ax.set_title(f'Direction-alignment hypothesis\nSpearman rho={rho:+.3f}, p={p:.4f}, n={len(clean)}')
ax.grid(alpha=0.3); ax.axhline(0, color='black', lw=0.5); ax.axvline(0, color='black', lw=0.5)

ax = axes[1]
ax.scatter(clean['magnitude_delta_B_solo'], clean['A_displaced'],
           s=80, alpha=0.7, color='#3366cc', edgecolors='black', lw=0.4)
ax.set_xlabel('|δB_solo|  (B-edit weight magnitude)')
ax.set_ylabel('A_displaced')
rho_mag, p_mag = stats.spearmanr(clean['magnitude_delta_B_solo'], clean['A_displaced'])
ax.set_title(f'Magnitude control\nSpearman rho={rho_mag:+.3f}, p={p_mag:.4f}')
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig1_cosine_vs_displaced.png', dpi=140)
plt.show()

### 17.8 Save

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {
    'notebook': 'Notebook 17 — Gradient Direction Hypothesis',
    'timestamp': ts, 'model': MODEL_NAME, 'n_pairs': len(pairs),
    'data_source': 'pair indices auto-fetched from github.com/rukmini-17/targeted-unlearning',
    'hypothesis': 'Damage to A is proportional to cosine(δA, δB_solo) in parameter space',
    'n_clean_rows': len(clean),
    'cosine_vs_A_displaced': {
        'spearman_rho': float(rho), 'spearman_p': float(p),
        'pearson_r': float(pearson_r), 'pearson_p': float(pearson_p),
    },
    'control_correlations': {
        'magnitude_delta_A_vs_A_displaced': {
            'rho': float(stats.spearmanr(clean['magnitude_delta_A'], clean['A_displaced'])[0]),
            'p':   float(stats.spearmanr(clean['magnitude_delta_A'], clean['A_displaced'])[1]),
        },
        'magnitude_delta_B_solo_vs_A_displaced': {
            'rho': float(rho_mag), 'p': float(p_mag),
        },
    },
    'multivariate_R2': float(r2),
    'mean_cosine': float(clean['cosine_delta_A_delta_B_solo'].mean()),
    'verdict': ('hypothesis_supported' if rho > 0.4 and p < 0.05
                else ('hypothesis_reversed' if rho < -0.4 and p < 0.05
                      else ('weak_effect' if p < 0.05 else 'hypothesis_not_supported'))),
}
with open(RESULTS_DIR / f'summary_nb17_{ts}.json', 'w') as f: json.dump(summary, f, indent=2)
df.to_csv(RESULTS_DIR / f'df_nb17_{ts}.csv', index=False)
print(json.dumps(summary, indent=2))

import shutil
bundle = f'nb17_results_{ts}'
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
from google.colab import files
files.download(f'{bundle}.zip')